# Jauge-05 — Twisteurs euclidiens : une sphère de structures complexes et le modèle standard

**Notebook** : Jauge-05

**Durée estimée** : 60 à 75 minutes.

**Prérequis** : [Jauge-04 — Des points aux spineurs : la relation d'incidence des twisteurs](Jauge-04-Points-Spineurs-Twisteurs-Python.ipynb) (spineurs, relation d'incidence), l'algèbre linéaire complexe de niveau licence (espaces propres, produits tensoriels, formes bilinéaires), et les quaternions — rappelés aux sections 1 et 2.

**Objectifs d'apprentissage** — à la fin de ce carnet, vous saurez :

1. construire la **sphère des structures complexes** d'un espace euclidien de dimension 4, et vérifier qu'une combinaison $aJ_1 + bJ_2 + cJ_3$ n'est une structure complexe que si $a^2 + b^2 + c^2 = 1$ ;
2. relier cette sphère au **fibré de Hopf** $S^3 \to S^2$ par la conjugaison quaternionique, et reconnaître la fibre comme une phase à droite ;
3. extraire les **spineurs de Weyl** comme espaces propres $\pm i$ d'une structure complexe, et comprendre pourquoi la puissance $\tfrac12$ de $\Lambda^2\mathbb{C}^2$ est **forcée** par la symétrie des poids ;
4. refaire le **compte de dimension** de l'espace des twisteurs projectif $\mathrm{PT} = \mathbb{CP}^3$, et distinguer le compte correct $6$ du compte naïf $5$ ;
5. reconstituer, sur une **génération** de fermions, la table des poids d'une théorie de jauge $\mathrm{SU}(4)$ et y retrouver l'hypercharge du modèle standard au facteur $2$ près.

**Source.** Ce carnet distille les sections 4.1 à 4.3 et 5 de **P. Woit, _Euclidean Twistor Unification_** (arXiv:[2104.05099](https://arxiv.org/abs/2104.05099)). La copie utilisée est archivée dans le gisement bibliographique du dépôt (rayon `Physics`) ; elle a été lue à la source, et aucune formule n'est reprise sans être **mesurée** ici. Les propositions qui ne se ramènent pas à un calcul sont citées comme telles, avec leur section.

**Ce que ce carnet ne fait pas.** Il ne dérive pas les représentations du groupe de spin, ne quantifie aucun champ sur $\mathrm{PT}$ et ne dit rien de la dynamique. Il mesure : les structures complexes de $\mathbb{R}^4$, la fibration de Hopf, les poids de $\mathrm{U}(1)$, les dimensions, et l'arithmétique d'une génération. C'est le socle géométrique de la proposition de Woit, pas la théorie.

**Convention de mesure.** Chaque section annonce son seuil **avant** de mesurer. Tout est calculé en `float64` et `complex128`, avec numpy seul, sans cadre d'apprentissage : le plancher numérique attendu est voisin de $10^{-15}$, et un résidu sous $10^{-12}$ se lit « exact à la précision machine ». Chaque affirmation vient avec un **contre-témoin construit pour échouer** — une mesure incapable d'échouer ne prouve rien.

**Plan.** 1. La sphère des structures complexes. 2. Le fibré de Hopf, et une seconde sphère de chiralité opposée. 3. Spineurs de Weyl et poids. 4. Le compte de dimension de $\mathrm{PT}$. 5. Une génération, seize états.


## 1. Une sphère de structures complexes sur $\mathbb{R}^4$

Une **structure complexe** sur $\mathbb{R}^4$ est une application linéaire $J$ telle que $J^2 = -\mathbb{1}$. Elle fait de $\mathbb{R}^4$ un espace vectoriel complexe de dimension 2, où $J$ joue le rôle de la multiplication par $i$.

Les quaternions en fournissent trois d'un coup. Sur $\mathbb{H} \cong \mathbb{R}^4$, la multiplication **à gauche** par $i$, $j$, $k$ est linéaire, et l'on note

$$J_1 = L(i), \qquad J_2 = L(j), \qquad J_3 = L(k) = J_1 J_2 .$$

La cellule suivante écrit ces trois matrices et vérifie les relations des quaternions. **Seuil annoncé : $10^{-12}$** — au-delà, une relation est fausse, pas « presque vraie ».


In [1]:
import numpy as np

# Les trois structures de base, dans l'ordre des coordonnees (e0, e1, e2, e3).
J1 = np.array([[0., -1., 0., 0.],
               [1.,  0., 0., 0.],
               [0.,  0., 0., -1.],
               [0.,  0., 1.,  0.]])
J2 = np.array([[0., 0., -1., 0.],
               [0., 0.,  0., 1.],
               [1., 0.,  0., 0.],
               [0., -1., 0., 0.]])
J3 = J1 @ J2
I4 = np.eye(4)

print("Imports OK : numpy", np.__version__)
print("J1^2 + I  : %.2e" % np.abs(J1 @ J1 + I4).max())
print("J2^2 + I  : %.2e" % np.abs(J2 @ J2 + I4).max())
print("J3^2 + I  : %.2e" % np.abs(J3 @ J3 + I4).max())
print("J1 J2 = J3 : %s | J2 J3 = J1 : %s | J3 J1 = J2 : %s"
      % (np.array_equal(J1 @ J2, J3), np.array_equal(J2 @ J3, J1),
         np.array_equal(J3 @ J1, J2)))


Imports OK : numpy 2.4.6
J1^2 + I  : 0.00e+00
J2^2 + I  : 0.00e+00
J3^2 + I  : 0.00e+00
J1 J2 = J3 : True | J2 J3 = J1 : True | J3 J1 = J2 : True


### Lecture du résultat : trois matrices, une algèbre

Les trois carrés sont **exactement nuls** en résidu (`0.00e+00`) : $J_1^2 = J_2^2 = J_3^2 = -\mathbb{1}$. Les trois produits tombent juste, y compris $J_3 = J_1 J_2$ : c'est l'algèbre des quaternions, écrite en matrices réelles $4 \times 4$ avec les coordonnées de $x = x_0 + x_1 i + x_2 j + x_3 k$ rangées dans l'ordre $(x_0, x_1, x_2, x_3)$.

Rien n'est approché ici : les matrices sont entières, les produits aussi. Le résidu nul n'est pas une performance numérique, c'est une identité.

La conséquence est une **sphère**. Pour des réels $a, b, c$, la matrice $J = aJ_1 + bJ_2 + cJ_3$ vérifie

$$J^2 = -(a^2 + b^2 + c^2)\,\mathbb{1},$$

parce que les trois matrices anticommutent. Elle n'est donc une structure complexe que sur la sphère unité $a^2 + b^2 + c^2 = 1$. La cellule suivante mesure cette équivalence dans les deux sens, avec deux contre-témoins.


In [2]:
rng = np.random.default_rng(42)

# 1000 directions unitaires sur la sphere, et la structure correspondante.
v = rng.normal(size=(1000, 3))
v /= np.linalg.norm(v, axis=1, keepdims=True)
Jv = (v[:, 0, None, None] * J1 + v[:, 1, None, None] * J2
      + v[:, 2, None, None] * J3)

print("sur la sphere : |J^2 + I| max = %.2e" % np.abs(Jv @ Jv + I4).max())
print("orthogonalite |J^T J - I| max = %.2e"
      % np.abs(Jv.transpose(0, 2, 1) @ Jv - I4).max())
print("determinant : %.12f .. %.12f"
      % (np.linalg.det(Jv).min(), np.linalg.det(Jv).max()))
print("norme de Frobenius : %.12f .. %.12f"
      % (np.linalg.norm(Jv, axis=(1, 2)).min(),
         np.linalg.norm(Jv, axis=(1, 2)).max()))


def coefficients(J: np.ndarray) -> np.ndarray:
    """Coordonnees de J sur la base (J1, J2, J3).

    Le produit scalaire est <A, B> = tr(A^T B) / 4, qui rend la base orthonormee.
    """
    return np.array([-np.trace(J @ Ji) / 4.0 for Ji in (J1, J2, J3)])


def auto_dual(J: np.ndarray) -> np.ndarray:
    """Composante auto-duale, base (e01 + e23, e02 - e13, e03 + e12)."""
    return np.array([0.5 * (J[0, 1] + J[2, 3]),
                     0.5 * (J[0, 2] - J[1, 3]),
                     0.5 * (J[0, 3] + J[1, 2])])


def anti_auto_dual(J: np.ndarray) -> np.ndarray:
    """Composante anti-auto-duale, base (e01 - e23, e02 + e13, e03 - e12)."""
    return np.array([0.5 * (J[0, 1] - J[2, 3]),
                     0.5 * (J[0, 2] + J[1, 3]),
                     0.5 * (J[0, 3] - J[1, 2])])


coef = np.stack([coefficients(J) for J in Jv])
print("relecture des coordonnees : %.2e" % np.abs(coef - v).max())
print("composante anti-auto-duale max : %.2e"
      % np.abs(np.stack([anti_auto_dual(J) for J in Jv])).max())

# Contre-temoins : deux matrices qui ressemblent a des structures complexes.
J_bad = J1 + 0.7 * J2
print("hors sphere (1, 0.7, 0) : |J^2 + I| = %.6f"
      % np.abs(J_bad @ J_bad + I4).max())
# Meme norme de Frobenius que les structures de la sphere (2), mais generique :
# antisymetrique et de norme 2 ne suffisent pas.
A = rng.normal(size=(4, 4))
A = A - A.T
A *= 2.0 / np.linalg.norm(A)
print("antisymetrique generique, norme 2 : |J^2 + I| = %.6f ; |J^T J - I| = %.6f"
      % (np.abs(A @ A + I4).max(), np.abs(A.T @ A - I4).max()))


sur la sphere : |J^2 + I| max = 4.44e-16
orthogonalite |J^T J - I| max = 4.44e-16
determinant : 1.000000000000 .. 1.000000000000
norme de Frobenius : 2.000000000000 .. 2.000000000000
relecture des coordonnees : 0.00e+00
composante anti-auto-duale max : 0.00e+00
hors sphere (1, 0.7, 0) : |J^2 + I| = 0.490000
antisymetrique generique, norme 2 : |J^2 + I| = 0.827768 ; |J^T J - I| = 0.827768


### Lecture du résultat : la sphère, et deux façons d'en sortir

Sur 1000 directions unitaires tirées au hasard (graine 42), tous les résidus sont au plancher machine :

| Mesure | Valeur |
|---|---|
| $|J^2 + \mathbb{1}|$ maximal | `4.44e-16` |
| orthogonalité $|J^{\mathsf T} J - \mathbb{1}|$ maximale | `4.44e-16` |
| déterminant | `1.000000000000` partout |
| norme de Frobenius | `2.000000000000` partout |
| relecture des coordonnées $(a,b,c)$ | `0.00e+00` |
| composante anti-auto-duale | `0.00e+00` |

Trois conséquences, toutes mesurées, aucune supposée. La norme de Frobenius vaut exactement 2 : la base $(J_1, J_2, J_3)$ est orthonormée pour le produit scalaire $\langle A, B\rangle = \tfrac14 \operatorname{tr}(A^{\mathsf T} B)$, et la sphère est de rayon 1 dans cette métrique. Le déterminant vaut $+1$ partout : ce sont des rotations d'ordre 4. Enfin la composante anti-auto-duale est nulle : **cette sphère est celle des structures auto-duales**, une des deux classes de chiralité de $\mathbb{R}^4$ — la section 2(d) montre l'autre.

Les deux contre-témoins échouent comme prévu, et de deux manières différentes :

- $J_1 + 0{,}7\,J_2$ sort de la sphère : $a^2 + b^2 + c^2 = 1{,}49$, donc $J^2 + \mathbb{1} = -0{,}49\,\mathbb{1}$ et le résidu vaut exactement `0.490000` — l'écart de $a^2+b^2+c^2$ à l'unité, et non une distance : la distance euclidienne de $(1;\,0{,}7;\,0)$ à la sphère vaut $0{,}22$ ;
- une matrice antisymétrique **générique de même norme de Frobenius** que les structures de la sphère — 2, et non 1 — n'est toujours pas une structure complexe : résidu `0.827768`, et l'écart à l'orthogonalité vaut le même nombre `0.827768`. Être antisymétrique et de norme 2 ne suffit pas ; il faut en plus $J^2 = -\mathbb{1}$, ce qui est une condition quadratique, pas linéaire.

Le second contre-témoin est le plus utile : il montre que la sphère n'est pas « l'ensemble des matrices antisymétriques de norme 2 », mais bien l'ensemble des structures complexes orthogonales d'une chiralité donnée.


## 2. De $S^3$ à $S^2$ : le fibré de Hopf

La sphère de la section 1 est de dimension 2. Les quaternions de norme 1 forment $S^3$, de dimension 3 : le passage de l'un à l'autre perd une dimension, et cette dimension perdue est une **phase**.

La multiplication à gauche par un quaternion unitaire $q$ est une isométrie de $\mathbb{H}$. Écrite en matrices réelles, elle forme le groupe $\mathrm{Sp}(1) \cong \mathrm{SU}(2)$, qui agit sur les structures complexes par **conjugaison** :

$$J \;\longmapsto\; L(q)\, J\, L(q)^{\mathsf T}, \qquad |q| = 1 .$$

Cette action envoie structure complexe sur structure complexe, et préserve la sphère. Partie de $J_3$ seule, elle produit une famille de structures dont les coordonnées se lisent explicitement : c'est la carte de Hopf, et la coordonnée que la matrice perd pour remonter d'un point à un quaternion est une phase.

La cellule suivante vérifie d'abord que $L(i)$, $L(j)$, $L(k)$ sont bien les trois matrices de la section 1, puis que toute unité imaginaire donne une structure complexe.


In [3]:
def gauche(q0: float, q1: float, q2: float, q3: float) -> np.ndarray:
    """Multiplication a gauche par q = q0 + q1 i + q2 j + q3 k, en 4x4 reel."""
    return np.array([[q0, -q1, -q2, -q3],
                     [q1,  q0, -q3,  q2],
                     [q2,  q3,  q0, -q1],
                     [q3, -q2,  q1,  q0]])


print("L(i) = J1 : %.2e | L(j) = J2 : %.2e | L(k) = J3 : %.2e"
      % (np.abs(gauche(0., 1., 0., 0.) - J1).max(),
         np.abs(gauche(0., 0., 1., 0.) - J2).max(),
         np.abs(gauche(0., 0., 0., 1.) - J3).max()))

# 1000 quaternions imaginaires purs unitaires -> 1000 structures complexes.
w = rng.normal(size=(1000, 3))
w /= np.linalg.norm(w, axis=1, keepdims=True)
Jw = np.stack([gauche(0.0, *wi) for wi in w])
print("quaternion imaginaire unitaire : |J^2 + I| max = %.2e"
      % np.abs(Jw @ Jw + I4).max())
print("orthogonalite max = %.2e"
      % np.abs(Jw.transpose(0, 2, 1) @ Jw - I4).max())
print("determinant : %.12f .. %.12f"
      % (np.linalg.det(Jw).min(), np.linalg.det(Jw).max()))
print("norme de Frobenius : %.12f .. %.12f"
      % (np.linalg.norm(Jw, axis=(1, 2)).min(),
         np.linalg.norm(Jw, axis=(1, 2)).max()))
print("composante anti-auto-duale max : %.2e"
      % np.abs(np.stack([anti_auto_dual(J) for J in Jw])).max())


L(i) = J1 : 0.00e+00 | L(j) = J2 : 0.00e+00 | L(k) = J3 : 0.00e+00
quaternion imaginaire unitaire : |J^2 + I| max = 4.44e-16
orthogonalite max = 4.44e-16
determinant : 1.000000000000 .. 1.000000000000
norme de Frobenius : 2.000000000000 .. 2.000000000000
composante anti-auto-duale max : 0.00e+00


### Lecture du résultat : tout quaternion imaginaire unitaire convient

Les trois identifications sont exactes (`0.00e+00`) : la multiplication à gauche reproduit les matrices de la section 1. Sur 1000 quaternions imaginaires purs unitaires, les résidus valent `4.44e-16` pour $J^2 + \mathbb{1}$ et pour l'orthogonalité, le déterminant vaut 1 et la norme 2 partout, et la composante anti-auto-duale reste nulle. Autrement dit, l'application

$$S^2 \ni (a, b, c) \;\longmapsto\; aJ_1 + bJ_2 + cJ_3$$

est la même chose que $q \mapsto L(q)$ restreinte aux quaternions imaginaires purs de norme 1. La sphère de la section 1 **est** la sphère des directions imaginaires.

Ce que cette cellule ne montre pas : le passage de $S^3$ entier à $S^2$. Pour cela il faut conjuguer, c'est-à-dire faire agir $q$ des deux côtés — et c'est là qu'apparaît la coordonnée de Hopf.


In [4]:
q = rng.normal(size=(500, 4))
q /= np.linalg.norm(q, axis=1, keepdims=True)
conj = np.stack([gauche(*qi) @ J3 @ gauche(*qi).T for qi in q])
coef_conj = np.stack([coefficients(J) for J in conj])


def hopf_k(q: np.ndarray) -> np.ndarray:
    """Coordonnees de Hopf de q k qbar, pour q unitaire (forme fermee)."""
    q0, q1, q2, q3 = q[:, 0], q[:, 1], q[:, 2], q[:, 3]
    return np.stack([2.0 * (q0 * q2 + q1 * q3),
                     2.0 * (q2 * q3 - q0 * q1),
                     q0 ** 2 + q3 ** 2 - q1 ** 2 - q2 ** 2], axis=1)


print("conjugaison : |J^2 + I| max = %.2e" % np.abs(conj @ conj + I4).max())
print("conjugaison : orthogonalite max = %.2e"
      % np.abs(conj.transpose(0, 2, 1) @ conj - I4).max())
print("conjugaison : norme du point de Hopf : %.12f .. %.12f"
      % (np.linalg.norm(coef_conj, axis=1).min(),
         np.linalg.norm(coef_conj, axis=1).max()))
print("ecart entre les coordonnees mesurees et la forme de Hopf : %.2e"
      % np.abs(coef_conj - hopf_k(q)).max())


conjugaison : |J^2 + I| max = 1.11e-15
conjugaison : orthogonalite max = 1.22e-15
conjugaison : norme du point de Hopf : 1.000000000000 .. 1.000000000000
ecart entre les coordonnees mesurees et la forme de Hopf : 2.22e-16


### Lecture du résultat : la conjugaison, et la carte de Hopf

La conjugaison préserve tout : $J^2 + \mathbb{1}$ reste sous `1.11e-15`, l'orthogonalité sous `1.22e-15`, et la norme des coordonnées vaut 1 partout. Elle est donc une action de $S^3$ sur la sphère.

Les coordonnées du point atteint ont une forme close, et l'écart entre la mesure et cette forme vaut `2.22e-16` :

$$q\, k\, \bar q \;=\; \bigl(\,2(q_0q_2 + q_1q_3),\;\; 2(q_2q_3 - q_0q_1),\;\; q_0^2 + q_3^2 - q_1^2 - q_2^2 \,\bigr).$$

C'est la **carte de Hopf** pour la direction de base $k$ : trois polynômes quadratiques en les quatre coordonnées de $q$, dont la somme des carrés vaut $(q_0^2+q_1^2+q_2^2+q_3^2)^2 = 1$. L'image est donc dans $S^2$, et la formule close est vérifiée numériquement sur 500 tirages. Ce carnet ne démontre pas la surjectivité : il établit la forme exacte de la carte, qui est ce dont la suite a besoin.

Il reste à trouver ce qui est perdu : deux quaternions distincts peuvent donner le même point. C'est la fibre, objet de la cellule suivante.


In [5]:
theta = rng.uniform(0.0, 2.0 * np.pi, size=500)
c, s = np.cos(theta), np.sin(theta)

# q multiplie a droite par e^{k theta} = (cos theta, 0, 0, sin theta)
comp_k = np.stack([q[:, 0] * c - q[:, 3] * s,
                   q[:, 1] * c + q[:, 2] * s,
                   q[:, 2] * c - q[:, 1] * s,
                   q[:, 0] * s + q[:, 3] * c], axis=1)

# Contre-temoin : la phase scalaire e^{i theta}
comp_i = np.stack([q[:, 0] * c - q[:, 1] * s,
                   q[:, 0] * s + q[:, 1] * c,
                   q[:, 2] * c + q[:, 3] * s,
                   q[:, 3] * c - q[:, 2] * s], axis=1)

print("fibre q e^(k theta) : ecart des points de Hopf = %.2e"
      % np.abs(hopf_k(comp_k) - hopf_k(q)).max())
print("fibre q e^(k theta) : normes %.12f .. %.12f"
      % (np.linalg.norm(comp_k, axis=1).min(),
         np.linalg.norm(comp_k, axis=1).max()))
J_k = np.stack([gauche(*comp_k[i]) @ J3 @ gauche(*comp_k[i]).T
                for i in range(500)])
print("fibre q e^(k theta) : ecart des structures = %.2e"
      % np.abs(J_k - conj).max())
ecart_i = np.abs(hopf_k(comp_i) - hopf_k(q))
print("contre-temoin q e^(i theta) : ecart max = %.4f" % ecart_i.max())

# Forme close de l'echec : la phase scalaire conjugue la direction k autour de i,
# d'un angle 2 theta. L'ecart vaut |sin(2 theta)| |h_j - h_k|, et ne s'annule
# donc que pour theta multiple de pi / 2.
hopf_j = np.stack([coefficients(gauche(*qi) @ J2 @ gauche(*qi).T) for qi in q])
forme_close = (np.cos(2 * theta)[:, None] * hopf_k(q)
               - np.sin(2 * theta)[:, None] * hopf_j)
print("contre-temoin q e^(i theta) : ecart a cos(2t) h_k - sin(2t) h_j = %.2e"
      % np.abs(hopf_k(comp_i) - forme_close).max())


fibre q e^(k theta) : ecart des points de Hopf = 3.33e-16


fibre q e^(k theta) : normes 1.000000000000 .. 1.000000000000
fibre q e^(k theta) : ecart des structures = 3.33e-16
contre-temoin q e^(i theta) : ecart max = 1.9728
contre-temoin q e^(i theta) : ecart a cos(2t) h_k - sin(2t) h_j = 4.44e-16


### Lecture du résultat : la fibre est une phase à droite

Multiplier $q$ **à droite** par $e^{k\theta} = \cos\theta + k\sin\theta$ ne change pas le point de Hopf : l'écart vaut `3.33e-16` sur les points comme sur les structures complexes, et les normes restent 1. La raison tient en une ligne, parce que $e^{k\theta}$ commute avec $k$ : $q\,e^{k\theta}\,k\,e^{-k\theta}\bar q = q\,k\,\bar q$. La phase à droite est donc dans le noyau, et elle forme un cercle $S^1$.

Le contre-témoin est une phase **scalaire** $e^{i\theta}$, également de norme 1, également un cercle. Elle ne convient pas : l'écart au point de départ monte à `1.97` sur 500 tirages. La raison est algébrique, et la cellule en donne la forme close. Puisque $k$ et $i$ ne commutent pas, conjuguer $k$ par $e^{i\theta}$ le fait tourner dans le plan engendré par $j$ et $k$ :

$$q\,e^{i\theta}\,k\,e^{-i\theta}\,\bar q \;=\; \cos(2\theta)\; q\,k\,\bar q \;-\; \sin(2\theta)\; q\,j\,\bar q ,$$

et l'écart mesuré à cette formule vaut `4.44e-16`. L'écart entre les deux points de Hopf est donc $|\sin 2\theta| \cdot |h_j(q) - h_k(q)|$ : il s'annule pour les multiples de $\pi/2$, pas avec $\theta$. C'est pourquoi le minimum observé sur 500 tirages est petit sans être nul, alors que la phase à droite donne zéro à la précision machine. La fibre du fibré est le sous-groupe qui **commute** avec la direction de base, pas un cercle quelconque.

D'où le fibré : $S^1 \to S^3 \to S^2$, et l'identification $S^2 \cong S^3/S^1 = \mathbb{CP}^1$. La sphère des structures complexes **est** la droite projective complexe — c'est ce que Woit appelle la fibre de $\mathrm{PT}$ au-dessus d'un point de $S^4$ (section 4.1).


In [6]:
sigma = (np.array([[0., 1.], [1., 0.]], dtype=complex),
         np.array([[0., -1j], [1j, 0.]], dtype=complex),
         np.array([[1., 0.], [0., -1.]], dtype=complex))


def carte(u: np.ndarray) -> np.ndarray:
    """Matrice 2x4 de x -> x . u, avec x = x0 - i (x1 s1 + x2 s2 + x3 s3).

    u est un vecteur complexe de norme 1 ; la carte associe a x de R^4 un
    element de C^2.
    """
    M = np.zeros((2, 4), dtype=complex)
    M[:, 0] = u
    for k in range(3):
        M[:, k + 1] = -1j * (sigma[k] @ u)
    return M


def vers_reel(M: np.ndarray) -> np.ndarray:
    """Matrice 4x4 reelle de la carte, coordonnees (Re z0, Im z0, Re z1, Im z1)."""
    R = np.zeros((4, 4))
    for k in range(4):
        z = M[:, k]
        R[:, k] = (z[0].real, z[0].imag, z[1].real, z[1].imag)
    return R


def structure_induite(u: np.ndarray) -> tuple[np.ndarray, np.ndarray]:
    """Structure transportee depuis C^2 par la carte x -> x . u, et la carte."""
    R = vers_reel(carte(u))
    return np.linalg.solve(R, J1 @ R), R


rng_u = np.random.default_rng(7)
u = rng_u.normal(size=(200, 2)) + 1j * rng_u.normal(size=(200, 2))
u /= np.linalg.norm(u, axis=1, keepdims=True)

J_ind, det_cartes = [], []
for ui in u:
    J, R = structure_induite(ui)
    J_ind.append(J)
    det_cartes.append(np.linalg.det(R))
J_ind = np.stack(J_ind)

print("u = (1, 0) : structure induite")
print(np.round(structure_induite(np.array([1.0, 0.0], dtype=complex))[0], 12))
print("200 droites : |J^2 + I| max = %.2e" % np.abs(J_ind @ J_ind + I4).max())
print("200 droites : orthogonalite max = %.2e"
      % np.abs(J_ind.transpose(0, 2, 1) @ J_ind - I4).max())
print("200 droites : determinant des cartes : %.12f .. %.12f"
      % (min(det_cartes), max(det_cartes)))
print("200 droites : composante auto-duale max = %.2e"
      % np.abs(np.stack([auto_dual(J) for J in J_ind])).max())
coef_ind = np.stack([coefficients(J) for J in J_ind])
recon = sum(coef_ind[:, i, None, None] * Ji
            for i, Ji in enumerate((J1, J2, J3)))
ecarts = np.abs(recon - J_ind).max(axis=(1, 2))
print("200 droites : ecart a la famille (J1, J2, J3) min/max = %.12f %.12f"
      % (ecarts.min(), ecarts.max()))


u = (1, 0) : structure induite


[[ 0.  0.  0.  1.]
 [-0. -0. -1. -0.]
 [ 0.  1.  0.  0.]
 [-1. -0. -0. -0.]]
200 droites : |J^2 + I| max = 4.44e-16
200 droites : orthogonalite max = 6.66e-16
200 droites : determinant des cartes : -1.000000000000 .. -1.000000000000
200 droites : composante auto-duale max = 1.67e-16
200 droites : ecart a la famille (J1, J2, J3) min/max = 0.600691577483 0.998028369613


### Lecture du résultat : la carte de la source, et la seconde chiralité

Jusqu'ici la sphère a été construite par les quaternions. La source en donne une autre route, et elle est **explicite** en section 4.1 : un point de $\mathrm{PT}$ est une droite $l \subset \mathbb{C}^4$ contenue dans $\mathbb{C}^2 = S_R$, et pour la droite engendrée par $(1, 0)$ la correspondance s'écrit

$$(x_0, x_1, x_2, x_3) \in \mathbb{R}^4 \;\longmapsto\; \bigl(x_0\,\mathbb{1} - i(x_1\sigma_1 + x_2\sigma_2 + x_3\sigma_3)\bigr)(1,0)^{\mathsf T} = (x_0 - ix_3,\; -ix_1 + x_2) \in \mathbb{C}^2 .$$

C'est cette carte que la cellule précédente applique — et pas seulement à $(1,0)$ : à toutes les droites $u \in \mathbb{CP}^1$ de la fibre — en transportant la multiplication par $i$ de $\mathbb{C}^2$ vers $\mathbb{R}^4$. Le résultat est net, et il ne coïncide pas avec la section 1 :

| Mesure sur 200 droites | Valeur |
|---|---|
| $|J^2 + \mathbb{1}|$ maximal | `4.44e-16` |
| orthogonalité maximale | `6.66e-16` |
| déterminant des cartes | `-1.000000000000` partout |
| composante **auto**-duale maximale | `1.67e-16` |
| écart à la famille $(J_1, J_2, J_3)$ | `0.600691577483` à `0.998028369613` |

La carte produit donc bien des structures complexes orthogonales — mais leur composante **auto**-duale est nulle, c'est-à-dire qu'elles sont **anti-auto-duales**, et leur écart à la famille quaternionique n'est jamais nul (minimum `0.60`). Ce sont d'autres structures : l'espace des structures complexes orthogonales de $\mathbb{R}^4$ a **deux composantes** connexes, deux sphères de même dimension et de chiralités opposées — fait classique, que la mesure illustre ici sans le démontrer. Le déterminant des cartes vaut $-1$ pour toutes les droites, ce qui est cohérent : une carte qui renverse l'orientation échange les deux classes.

Pour $u = (1, 0)$ la structure obtenue s'écrit

$$J = \begin{pmatrix} 0 & 0 & 0 & 1\\ 0 & 0 & -1 & 0\\ 0 & 1 & 0 & 0\\ -1 & 0 & 0 & 0\end{pmatrix},$$

soit la 2-forme $e^{03} - e^{12}$ : une structure parfaitement légitime, simplement de l'autre chiralité.

**Ce que le carnet retient de ce contre-témoin.** La suite travaille avec la composante quaternionique, parce que c'est elle que les quaternions fabriquent et que la carte de Hopf décrit. Le carnet ne prétend pas que l'autre composante est identique, ni qu'un choix de coordonnées est neutre : il nomme les deux, mesure l'écart, et note que la carte que la source écrit pour la droite $(1,0)$ tombe, dans la convention de la section 1, du côté **anti-auto-dual**. Cette convention est celle des fonctions `auto_dual` et `anti_auto_dual`, dont la base est $(e^{01}+e^{23},\, e^{02}-e^{13},\, e^{03}+e^{12})$ pour la partie auto-duale ; un choix opposé échangerait les étiquettes sans changer une seule des mesures.


### Exercice 1 — reconstruire un point depuis le fibré

La carte de Hopf va de $S^3$ vers $S^2$. Le problème inverse — trouver un quaternion unitaire $q$ dont l'image soit un point donné $(a, b, c)$ de la sphère — a une solution explicite, et c'est ce que vous allez écrire.

**Objectif.** Pour le point $(a, b, c) = (0{,}6,\; -0{,}48,\; 0{,}64)$, construire un quaternion unitaire $q$ tel que `hopf_k(q)` redonne ce point à $10^{-12}$ près.

**Contexte.** La fonction `hopf_k` est définie à la section 2(b) et attend un tableau de forme `(N, 4)`. Une famille de solutions est indexée par un cercle entier : il suffit d'en trouver une.

**Indice.** Poser $q_3 = 0$ suffit tant que $c > -1$. La troisième coordonnée de la carte donne alors $q_0^2 = (1 + c)/2$, et les deux autres coordonnées fixent $q_1$ et $q_2$ une fois $q_0$ choisi.

La cellule s'exécute sans erreur tant que l'exercice n'est pas complété.


In [7]:
# Exercice 1 — reconstruire un point de la sphere depuis le fibre de Hopf
# TODO etudiant : etant donne un point (a, b, c) de la sphere unite (c > -1),
# construire un quaternion unitaire q dont l'image de Hopf soit ce point, puis
# verifier hopf_k(q) == (a, b, c) a 1e-12 pres.
# Indice : poser q3 = 0 suffit pour c > -1 ; la troisieme coordonnee donne
# alors q0^2 = (1 + c) / 2, et les deux autres fixent q1 et q2.
point_ex1 = np.array([0.6, -0.48, 0.64])   # a^2 + b^2 + c^2 = 1
q_ex1 = None  # TODO etudiant : quaternion (q0, q1, q2, q3)
if q_ex1 is not None:
    hopf_mesure = hopf_k(np.asarray([q_ex1], dtype=float))[0]
    ecart = np.abs(hopf_mesure - point_ex1).max()
    print("ecart du point reconstruit : %.2e" % ecart)
else:
    print("Exercice a completer")


Exercice a completer


## 3. Spineurs de Weyl et poids : d'où vient la puissance $\tfrac12$

Fixons une structure complexe de la famille, $J_3$, et regardons $\mathbb{C}^4$ comme un espace complexe muni de $J_3$. Les valeurs propres de $J_3$ sont $\pm i$ ; chacune a une multiplicité, et les espaces propres correspondants sont les **spineurs**.

Sur un espace euclidien de dimension 4, le groupe des rotations est $\mathrm{Spin}(4) \cong \mathrm{SU}(2) \times \mathrm{SU}(2)$, et ses deux facteurs agissent sur deux représentations complexes de dimension 2, les **spineurs de Weyl** $S_L$ et $S_R$. La cellule suivante vérifie que les espaces propres de $J_3$ ont exactement cette structure, et que la conjugaison complexe échange les deux.

Vient ensuite la question des poids. Si un cercle $\mathrm{U}(1)$ agit sur $\mathbb{C}^2$ par le scalaire $e^{i\theta}$, il agit sur le produit tensoriel $\Lambda^\bullet \mathbb{C}^2$ avec des poids entiers : $0$ sur $\Lambda^0$, $1$ sur $\Lambda^1$, $2$ sur $\Lambda^2$. La section mesure ces poids, puis montre qu'un seul décalage entier les symétrise, et que ce décalage est exactement la demi-puissance du fibré déterminant.


In [8]:
valeurs, vecteurs = np.linalg.eig(J3.astype(complex))
plus = [k for k in range(4) if abs(valeurs[k] - 1j) < 1e-9]
moins = [k for k in range(4) if abs(valeurs[k] + 1j) < 1e-9]
Vp, Vm = vecteurs[:, plus], vecteurs[:, moins]

print("multiplicites des valeurs propres +i / -i : %d / %d"
      % (len(plus), len(moins)))
print("rang [C+, C-] = %d ; |det [C+, C-]| = %.3f"
      % (np.linalg.matrix_rank(np.hstack([Vp, Vm]), tol=1e-9),
         abs(np.linalg.det(np.hstack([Vp, Vm])))))
print("rang [conj(C+), C-] = %d ; |det [conj(C+), C-]| = %.3e"
      % (np.linalg.matrix_rank(np.hstack([Vp.conj(), Vm]), tol=1e-9),
         abs(np.linalg.det(np.hstack([Vp.conj(), Vm])))))
print("|J v - i v| sur C+ : %.2e ; |J conj(v) + i conj(v)| : %.2e"
      % (np.abs(J3 @ Vp - 1j * Vp).max(),
         np.abs(J3 @ Vp.conj() + 1j * Vp.conj()).max()))
print("rang [Re C+, Im C+] = %d "
      "(4 = C+ ne contient aucun vecteur reel non nul)"
      % np.linalg.matrix_rank(np.hstack([Vp.real, Vp.imag]), tol=1e-9))


multiplicites des valeurs propres +i / -i : 2 / 2
rang [C+, C-] = 4 ; |det [C+, C-]| = 1.000
rang [conj(C+), C-] = 2 ; |det [conj(C+), C-]| = 2.778e-32
|J v - i v| sur C+ : 1.11e-16 ; |J conj(v) + i conj(v)| : 1.11e-16
rang [Re C+, Im C+] = 4 (4 = C+ ne contient aucun vecteur reel non nul)


### Lecture du résultat : deux plans complexes conjugués

Les multiplicités sont `2 / 2` : chaque valeur propre $\pm i$ a un espace propre de dimension complexe 2, comme attendu pour les spineurs de Weyl. La réunion des deux bases est de rang 4, et son déterminant vaut 1 en module : les quatre colonnes forment une base de $\mathbb{C}^4$.

La conjugaison complexe fait passer d'un espace à l'autre, et le montre deux fois : le rang de $[\overline{C_+}, C_-]$ vaut 2 (les quatre colonnes sont liées, l'espace engendré est exactement $C_-$), et le déterminant correspondant est nul à `2.778e-32` près. Côté vecteurs, $J v = i v$ sur $C_+$ et $J \bar v = -i \bar v$ : les deux résidus valent `1.11e-16`.

Enfin, $C_+$ ne contient **aucun vecteur réel non nul**. La mesure est le rang de la matrice $4 \times 4$ réelle $[\operatorname{Re} C_+, \operatorname{Im} C_+]$, qui vaut 4 : ces quatre colonnes forment une base de $\mathbb{R}^4$, donc tout vecteur réel est la partie réelle d'un vecteur de $C_+$. Si un vecteur réel non nul appartenait lui-même à $C_+$, sa conjuguée — qui est lui — appartiendrait à $C_-$, et il vivrait donc dans $C_+ \cap C_-$ : or $[C_+, C_-]$ est de rang 4, donc $C_+ \oplus C_- = \mathbb{C}^4$ et l'intersection est nulle. Un spineur de Weyl est donc réellement un objet complexe : il n'existe pas de spineur de Weyl réel, et c'est pourquoi la conjugaison complexe est une opération physique — elle échange deux représentations distinctes, elle ne les identifie pas.

C'est la structure que Woit note en section 4.1 : les spineurs se scindent en $S_L = \mathbb{C}^2_0$ et $S_R = \mathbb{C}_{-1} \oplus \mathbb{C}_{+1}$. Deux précautions de notation. D'abord, $C_\pm$ désigne ici les espaces propres des valeurs propres $\pm i$ de $J_3$ ; les indices de la source, $\mathbb{C}_{-1}$ et $\mathbb{C}_{+1}$, sont des **poids** sous le $\mathrm{U}(1)$ de la section 3(b) — ce n'est pas la même chose. Ensuite, le calcul ne distingue pas $S_L$ de $S_R$ : c'est la section 3(b) qui fixe l'étiquette, en attachant $S_L$ à la partie impaire $\Lambda^1\mathbb{C}^2$ (poids $0$) et $S_R$ aux parties $\Lambda^0 \oplus \Lambda^2$ (poids $\mp 1$), conformément à l'équation (4.1) de la source.


In [9]:
theta_s = 0.37
g = np.exp(1j * theta_s)
poids = np.array([1.0, g, g, g * g])        # Lambda^0, Lambda^1 (x2), Lambda^2
twiste = poids / np.sqrt(g * g)

print("poids de Lambda*C^2 :", np.round(np.angle(poids) / theta_s, 12))
print("poids apres torsion par (Lambda^2)^(-1/2) :",
      np.round(np.angle(twiste) / theta_s, 12))
for decalage in (-2, -1, 0, 1):
    essai = np.sort(np.angle(poids) / theta_s) + decalage
    print("decalage %+d : asymetrie |min + max| = %.6f"
          % (decalage, np.abs(essai.min() + essai.max())))
wL = np.sort(np.angle(poids) / theta_s)
wT = np.sort(np.angle(twiste) / theta_s)
print("asymetrie : Lambda*C^2 %.6f / twiste %.6f ; somme des poids %.6f"
      % (np.abs(wL + wL[::-1]).max(), np.abs(wT + wT[::-1]).max(), wL.sum()))


poids de Lambda*C^2 : [0. 1. 1. 2.]
poids apres torsion par (Lambda^2)^(-1/2) : [-1.  0.  0.  1.]
decalage -2 : asymetrie |min + max| = 2.000000
decalage -1 : asymetrie |min + max| = 0.000000
decalage +0 : asymetrie |min + max| = 2.000000
decalage +1 : asymetrie |min + max| = 4.000000
asymetrie : Lambda*C^2 2.000000 / twiste 0.000000 ; somme des poids 4.000000


### Lecture du résultat : la symétrie force la demi-puissance

Sous l'action du scalaire $e^{i\theta}$ avec $\theta = 0{,}37$, les poids de $\Lambda^\bullet\mathbb{C}^2$ se lisent `[0, 1, 1, 2]` : zéro sur $\Lambda^0$, un sur chacun des deux vecteurs de base de $\Lambda^1$, deux sur $\Lambda^2$. La somme vaut 4, la dimension de l'espace.

Ce jeu de poids n'est **pas symétrique** par rapport à zéro : $\min + \max = 0 + 2 = 2$, et l'asymétrie mesurée vaut `2.000000`. Décaler tous les poids d'un entier $k$ donne $\{k,\; 1+k,\; 1+k,\; 2+k\}$, d'asymétrie $|k + (2 + k)| = |2 + 2k|$ : elle ne s'annule que pour $k = -1$. Le balayage le confirme — `2.000000` pour $k = -2$, `0.000000` pour $k = -1$, `2.000000` pour $k = 0`, `4.000000` pour $k = +1$ — et l'équation dit déjà que ce zéro est **unique** : la mesure le vérifie, elle ne le démontre pas.

Twister par $(\Lambda^2\mathbb{C}^2)^{-1/2}$ — c'est-à-dire diviser chaque poids par $e^{i\theta}$, la puissance $\tfrac12$ du déterminant — décale tous les poids de $-1$ et donne `[-1, 0, 0, 1]`, dont l'asymétrie est `0.000000`. Le seul décalage qui symétrise le spectre est donc $-1$, et $-1$ est la moitié du poids de $\Lambda^2\mathbb{C}^2$ : la **demi-puissance** du fibré déterminant est le seul exposant constant qui rende le spectre symétrique.

C'est le contenu de l'équation (4.1) de Woit, notée $S_R \oplus S_L = \Lambda^*\mathbb{C}^2 \otimes (\Lambda^2\mathbb{C}^2)^{1/2}$, avec $S_R = (\Lambda^0\mathbb{C}^2 \oplus \Lambda^2\mathbb{C}^2) \otimes (\Lambda^2\mathbb{C}^2)^{1/2}$ et $S_L = \Lambda^1\mathbb{C}^2 \otimes (\Lambda^2\mathbb{C}^2)^{1/2}$ : la partie **paire** donne $S_R$, la partie **impaire** donne $S_L$, et l'on retrouve $S_L = \mathbb{C}^2_0$ et $S_R = \mathbb{C}_{-1} \oplus \mathbb{C}_{+1}$. Une réserve : la convention de signe du $\mathrm{U}(1)$ fait que le carnet écrit la torsion comme la puissance $-1/2$ là où la source écrit $+1/2$. Ce que la mesure établit, indépendamment de cette convention, c'est l'exposant **moitié** et le partage des poids $\{0, 0\}$ / $\{-1, +1\}$ entre $S_L$ et $S_R$. La puissance $\tfrac12$ n'est pas un ornement : sans elle, deux des quatre poids restent du même côté de zéro.


### Exercice 2 — le projecteur sur un spineur de Weyl

La section 3 a obtenu les deux spineurs comme espaces propres de $J_3$. On peut les isoler sans jamais diagonaliser : un projecteur spectral s'écrit directement avec la matrice et ses valeurs propres.

**Objectif.** Construire la matrice $P_+$ (de taille $4 \times 4$, à coefficients complexes) qui projette sur l'espace propre $C_+$ de $J_3$ — celui de la valeur propre $+i$. La cellule vérifie ensuite quatre propriétés : l'idempotence $P_+^2 = P_+$, l'annihilation de $C_-$, le rang 2, et l'action identité sur $C_+$.

**Contexte.** Les espaces propres `Vp` (pour $C_+$) et `Vm` (pour $C_-$) sont en mémoire, ainsi que `J3` et `I4`. Les quatre mesures doivent tomber au plancher machine ($10^{-15}$) sauf le rang, qui est un entier.

**Indice.** Deux valeurs propres seulement, $\lambda_+ = +i$ et $\lambda_- = -i$ : l'interpolation de Lagrange sur ces deux points est la formule des projecteurs spectraux, et elle ne demande que $J_3$ et l'identité. Une normalisation au hasard, ou le choix de la mauvaise valeur propre, se voit sur les mesures.

La cellule s'exécute sans erreur tant que l'exercice n'est pas complété.


In [10]:
# Exercice 2 — le projecteur sur un spineur de Weyl
# TODO etudiant : construire la matrice P_plus (4x4, complexe) qui projette sur
# l'espace propre C+ de J3 (valeur propre +i), puis lancez la cellule : elle
# mesure l'idempotence, l'annihilation sur C-, le rang et l'action sur C+.
# Indice : un projecteur spectral s'ecrit avec J3, l'identite et les deux
# valeurs propres — pas avec les vecteurs propres (interpolation de Lagrange).
P_plus = None  # TODO etudiant : matrice 4x4 complexe
if P_plus is not None:
    print("idempotence |P+^2 - P+| = %.2e"
          % np.abs(P_plus @ P_plus - P_plus).max())
    print("action sur C- : |P+ v| max = %.2e" % np.abs(P_plus @ Vm).max())
    print("rang de P+ : %d" % np.linalg.matrix_rank(P_plus, tol=1e-9))
    print("action sur C+ : |P+ v - v| max = %.2e"
          % np.abs(P_plus @ Vp - Vp).max())
else:
    print("Exercice a completer")


Exercice a completer


## 4. Le compte de dimension de $\mathrm{PT} = \mathbb{CP}^3$

Woit construit $\mathrm{PT}$ comme un fibré en droites projectives complexes au-dessus de $S^4$ : c'est l'espace des twisteurs projectifs, de dimension réelle 6, dont la fibre $\mathbb{CP}^1$ est la sphère de la section 2 et la base $S^4 = \mathbb{HP}^1$ est la sphère des droites quaternioniques.

Le compte de dimension s'écrit $\mathrm{PT} = \mathrm{SU}(4)/\mathrm{S}(\mathrm{U}(1)\times\mathrm{U}(3))$ — la source donne les trois écritures équivalentes $\mathrm{U}(4)/(\mathrm{U}(1)\times\mathrm{U}(3)) = \mathrm{SU}(4)/\mathrm{S}(\mathrm{U}(1)\times\mathrm{U}(3)) = \mathrm{SU}(4)/\mathrm{U}(3)$, qui ont toutes la même dimension. La cellule suivante refait le compte à partir des bases de matrices, sans supposer le résultat : elle mesure la dimension de $\mathfrak{su}(4)$, celle du stabilisateur, et leur différence.

L'intérêt du calcul est un piège. Le compte **naïf** — $\dim\mathfrak{su}(4) - \dim(\mathfrak{u}(1)\oplus\mathfrak{u}(3))$ — ne donne pas 6 mais 5, un nombre impair, qui ne peut être la dimension d'aucune variété complexe. L'erreur vient de la condition de déterminant : elle porte des deux côtés de la fraction, et l'oublier d'un seul côté coûte une dimension.


In [11]:
def base_u(n: int) -> list[np.ndarray]:
    """Base anti-hermitienne de u(n) : n diagonales, puis n(n-1)/2 paires."""
    mats = []
    for j in range(n):
        E = np.zeros((n, n), dtype=complex)
        E[j, j] = 1j
        mats.append(E)
    for j in range(n):
        for k in range(j + 1, n):
            E = np.zeros((n, n), dtype=complex)
            E[j, k], E[k, j] = 1.0, -1.0
            mats.append(E)
            F_ = np.zeros((n, n), dtype=complex)
            F_[j, k], F_[k, j] = 1j, 1j
            mats.append(F_)
    return mats


def rang(mats: list[np.ndarray]) -> int:
    """Dimension reelle de l'espace engendre, lue sur la matrice de Gram."""
    G = np.array([[np.trace(A.conj().T @ B).real for B in mats] for A in mats])
    return np.linalg.matrix_rank(G, tol=1e-9)


def contrainte_trace(mats: list[np.ndarray]) -> int:
    """Rang de l'application trace sur l'espace engendre (0 ou 1)."""
    tr = np.array([[np.trace(M).real, np.trace(M).imag] for M in mats])
    return np.linalg.matrix_rank(tr, tol=1e-9)


u4 = base_u(4)
dim_u4 = rang(u4)
dim_su4 = dim_u4 - contrainte_trace(u4)

amb = []
E = np.zeros((4, 4), dtype=complex)
E[0, 0] = 1j
amb.append(E)
for B in base_u(3):
    M = np.zeros((4, 4), dtype=complex)
    M[1:, 1:] = B
    amb.append(M)
dim_amb = rang(amb)
dim_s = dim_amb - contrainte_trace(amb)

print("dim u(4) = %d ; contrainte de trace = %d ; dim su(4) = %d"
      % (dim_u4, contrainte_trace(u4), dim_su4))
print("dim u(1) + u(3) = %d ; contrainte de trace = %d ; dim S(U(1)xU(3)) = %d"
      % (dim_amb, contrainte_trace(amb), dim_s))
print("dim PT = %d - %d = %d" % (dim_su4, dim_s, dim_su4 - dim_s))
print("decomposition 4 + 2 = %d ; dimension complexe %d"
      % (4 + 2, (dim_su4 - dim_s) // 2))
print("compte naif sans contrainte de trace : %d - %d = %d (impair)"
      % (dim_su4, dim_amb, dim_su4 - dim_amb))


dim u(4) = 16 ; contrainte de trace = 1 ; dim su(4) = 15
dim u(1) + u(3) = 10 ; contrainte de trace = 1 ; dim S(U(1)xU(3)) = 9
dim PT = 15 - 9 = 6
decomposition 4 + 2 = 6 ; dimension complexe 3
compte naif sans contrainte de trace : 15 - 10 = 5 (impair)


### Lecture du résultat : où le compte naïf se trompe

Le calcul se déroule en trois temps :

| Étape | Mesure |
|---|---|
| $\dim \mathfrak{u}(4)$, puis contrainte de trace | `16`, puis `1`, donc $\dim\mathfrak{su}(4) = 15$ |
| $\dim(\mathfrak{u}(1)\oplus\mathfrak{u}(3))$, puis même contrainte | `10`, puis `1`, donc $\dim \mathrm{S}(\mathrm{U}(1)\times\mathrm{U}(3)) = 9$ |
| différence | $15 - 9 = 6$ |

La contrainte de trace est de rang 1 dans les deux cas : elle retire exactement une dimension, et c'est la même dans $\mathrm{SU}(4)$ (où elle force $\det = 1$) et dans le stabilisateur (où elle force $\det\alpha \cdot \det A = 1$). Le compte naïf omet la seconde : il donne `15 - 10 = 5`, impair, donc impossible pour un espace complexe de dimension 3.

Le résultat correct se lit aussi en termes géométriques : `6 = 4 + 2`, la dimension de la base $S^4$ plus celle de la fibre $\mathbb{CP}^1$, et `3` en dimension complexe. C'est $\mathbb{CP}^3$, l'espace projectif des droites de $\mathbb{C}^4$ — et $\mathrm{SU}(4)$ y agit transitivement, le stabilisateur d'une droite étant précisément $\mathrm{S}(\mathrm{U}(1)\times\mathrm{U}(3))$.

Le contre-témoin est intégré au calcul : sans la contrainte de trace, la différence vaut 5. Une seule condition oubliée suffit donc à produire un objet qui n'existe pas comme variété complexe — le genre d'erreur qu'un compte de dimension doit rendre visible.


## 5. Une génération, seize états

Il reste la partie la plus frappante de la proposition : sur $\mathrm{PT} = \mathrm{SU}(4)/\mathrm{S}(\mathrm{U}(1)\times\mathrm{U}(3))$, une **génération** de fermions se lit dans un seul objet,

$$\operatorname{Hom}\bigl(l \oplus l^\perp,\; S_R \oplus S_L\bigr),$$

où $l \subset \mathbb{C}^4$ est une droite, $l^\perp = \mathbb{C}^4/l$ le fibré quotient de dimension complexe 3, et $S_R, S_L$ les deux fibrés de spineurs de la section 3.

Le point technique est le poids. Si $l$ porte le poids $w_l$ sous le $\mathrm{U}(1)$ du stabilisateur, la condition de trace sur $\mathbb{C}^4$ impose

$$w_l + 3\,w_\perp = 0,$$

donc $w_\perp = -w_l/3$ : le poids de $l^\perp$ est **forcé** par celui de $l$. La source l'énonce dans les mêmes termes — « if it acts with weight 1 on $l$, it will act with weight $-1/3$ on $l^\perp$ » (section 4.2). C'est la même condition de déterminant que la section 4, vue cette fois sur les poids.

Le dual d'une droite de poids $w$ porte le poids $-w$, et un $\operatorname{Hom}$ est un produit tensoriel par le dual : chaque multiplet reçoit le poids $-w$ de sa droite, plus le poids du spineur. Les spineurs, eux, portent leurs poids de la section 3 : $0$ pour $S_L$, et $\mp 1$ pour les deux composantes de $S_R$.

La cellule suivante dresse la table, en fractions exactes — pas en flottants, parce que les rapports de la section suivante doivent tomber juste.


In [12]:
from fractions import Fraction

w_l = Fraction(1)
w_perp = -w_l / 3
etats = [
    ("(nu_e, e)_L", "l* x S_L", 2, w_l * -1 + Fraction(0)),
    ("e_R", "l* x S_R", 1, w_l * -1 + Fraction(-1)),
    ("(nu_e)_R", "l* x S_R", 1, w_l * -1 + Fraction(1)),
    ("(u, d)_L", "(l^perp)* x S_L", 6, w_perp * -1 + Fraction(0)),
    ("d_R", "(l^perp)* x S_R", 3, w_perp * -1 + Fraction(-1)),
    ("u_R", "(l^perp)* x S_R", 3, w_perp * -1 + Fraction(1)),
]

print("poids de l sur C^4 : %s ; poids de l^perp : %s" % (w_l, w_perp))
print("contrainte de trace sur C^4 : 1*(%s) + 3*(%s) = %s"
      % (w_l, w_perp, 1 * w_l + 3 * w_perp))
total = 0
for nom, origine, dim, poids in etats:
    total += dim
    print("%-12s %-18s poids %-6s dim %d" % (nom, origine, poids, dim))
print("total : %d etats ; (1 + 3)(2 + 2) = %d" % (total, (1 + 3) * (2 + 2)))


poids de l sur C^4 : 1 ; poids de l^perp : -1/3
contrainte de trace sur C^4 : 1*(1) + 3*(-1/3) = 0
(nu_e, e)_L  l* x S_L           poids -1     dim 2
e_R          l* x S_R           poids -2     dim 1
(nu_e)_R     l* x S_R           poids 0      dim 1
(u, d)_L     (l^perp)* x S_L    poids 1/3    dim 6
d_R          (l^perp)* x S_R    poids -2/3   dim 3
u_R          (l^perp)* x S_R    poids 4/3    dim 3
total : 16 etats ; (1 + 3)(2 + 2) = 16


### Lecture du résultat : la table, et la contrainte de trace

La table compte seize états complexes, et le total se vérifie de deux façons indépendantes : la somme des dimensions des six multiplets (`2+1+1+6+3+3`) et le produit $(1+3)(2+2)$, qui est la dimension de $l \oplus l^\perp$ fois celle de $S_R \oplus S_L$. Les deux valent `16`.

| Multiplet | Origine | Poids | Dimension |
|---|---|---|---|
| $(\nu_e, e)_L$ | $l^* \otimes S_L$ | $-1$ | 2 |
| $e_R$ | $l^* \otimes S_R$ | $-2$ | 1 |
| $(\nu_e)_R$ | $l^* \otimes S_R$ | $0$ | 1 |
| $(u, d)_L$ | $(l^\perp)^* \otimes S_L$ | $1/3$ | 6 |
| $d_R$ | $(l^\perp)^* \otimes S_R$ | $-2/3$ | 3 |
| $u_R$ | $(l^\perp)^* \otimes S_R$ | $4/3$ | 3 |

Les noms de multiplets sont ceux du modèle standard, et c'est là que la construction devient intéressante : les dimensions et les poids tombent sur les multiplets connus. Le doublet $(\nu_e, e)_L$ a deux états, le singulet $e_R$ un seul, la paire de quarks $(u, d)_L$ en a six (trois couleurs fois deux), et les singulets de quarks trois chacun. La source énumère les six termes correspondants en section 4.2, à une inversion près — elle place $u_R$ avant $d_R$ ; la table suit l'ordre du code, et les correspondances sont terme à terme.

La contrainte de trace se vérifie exactement : $1 \cdot (1) + 3 \cdot (-1/3) = 0$. C'est elle qui fixe $w_\perp$, et donc tous les poids du secteur des quarks. Retirer cette contrainte ne laisse pas la table indéterminée : elle la rend fausse, comme la section suivante le mesure.


In [13]:
Y_sm = [Fraction(-1, 2), Fraction(-1), Fraction(0),
        Fraction(1, 6), Fraction(-1, 3), Fraction(2, 3)]
rapports = [poids / y for (_, _, _, poids), y in zip(etats, Y_sm) if y != 0]
print("hypercharges Y_SM :", [str(y) for y in Y_sm])
print("rapports poids / Y_SM :", [str(r) for r in rapports])
print("ecart maximal au facteur 2 :", max(abs(r - 2) for r in rapports))

w_perp_bad = Fraction(-1, 2)
poids_lepton = [w_l * -1 + Fraction(0), w_l * -1 + Fraction(-1),
                w_l * -1 + Fraction(1)]
poids_quark = [w_perp_bad * -1 + s for s in (Fraction(0), Fraction(-1),
                                             Fraction(1))]
poids_bad = poids_lepton + poids_quark
rapports_bad = [p / y for p, y in zip(poids_bad, Y_sm) if y != 0]
ecart_bad = float(max(abs(r - 2) for r in rapports_bad))
print("contre-temoin w_perp = -1/2 : poids", [str(p) for p in poids_bad])
print("contre-temoin : rapports", [str(r) for r in rapports_bad],
      "| ecart maximal au facteur 2 :", ecart_bad)


hypercharges Y_SM : ['-1/2', '-1', '0', '1/6', '-1/3', '2/3']
rapports poids / Y_SM : ['2', '2', '2', '2', '2']
ecart maximal au facteur 2 : 0
contre-temoin w_perp = -1/2 : poids ['-1', '-2', '0', '1/2', '-1/2', '3/2']
contre-temoin : rapports ['2', '2', '3', '3/2', '9/4'] | ecart maximal au facteur 2 : 1.0


### Lecture du résultat : un facteur 2, et un contre-témoin qui le détecte

En fractions exactes, les rapports des cinq poids non nuls à l'hypercharge conventionnelle valent tous `2` :

$$-1 \big/ -\tfrac12 = 2, \quad -2 \big/ -1 = 2, \quad \tfrac13 \big/ \tfrac16 = 2, \quad -\tfrac23 \big/ -\tfrac13 = 2, \quad \tfrac43 \big/ \tfrac23 = 2 .$$

L'écart maximal au facteur 2 est `0`. Les poids de $\mathrm{U}(1)$ de cette construction sont donc **deux fois** l'hypercharge du modèle standard dans la convention où la charge s'écrit $Q = T_3 + Y$ — autrement dit, ce sont ces poids eux-mêmes dans la convention $Q = T_3 + Y/2$, celle que la source appelle « weak hypercharge » (section 4.2). Ce facteur 2 est une normalisation, pas une prédiction : il dépend de la convention choisie pour $Y$, et le carnet ne lui attribue aucune signification physique.

Le contre-témoin montre que l'arithmétique discrimine. En prenant $w_\perp = -1/2$ au lieu de $-1/3$ — c'est-à-dire en violant la contrainte de trace de $\mathbb{C}^4$ — les poids deviennent `[-1, -2, 0, 1/2, -1/2, 3/2]` et les rapports `[2, 2, 3, 3/2, 9/4]` : l'écart maximal au facteur 2 monte à `1.0`. Une seule valeur fausse casse trois des cinq rapports. La cohérence de la table n'est donc pas un artefact de lecture : elle dépend d'une condition vérifiable.

**Ce que cette section établit, et ce qu'elle n'établit pas.** Elle établit que, sur $\mathrm{PT}$, l'assignation naturelle des poids à une droite et à son orthogonal reproduit exactement les dimensions et les poids d'une génération du modèle standard, à une normalisation près. Elle n'établit pas que la nature choisisse cette construction, ni d'où viennent les générations — la source ouvre la section 5 sur « it is unclear what the origin of generations might be ».


### Exercice 3 — localiser l'alignement

La contrainte de trace fixe $w_\perp$ à partir de $w_l$. Vous allez la retrouver par balayage, sans la lui donner.

**Objectif.** Balayer $w_\perp = k/12$ pour $k$ de $-12$ à $12$, recalculer les six poids de la table, et trouver l'unique $k$ pour lequel l'écart maximal au facteur 2 est nul.

**Contexte.** Les listes `etats` et `Y_sm` de la section 5 sont en mémoire, ainsi que `w_l` et la fonction de construction des poids. Le balayage doit être exact : travaillez en `Fraction`, pas en flottants, sinon le zéro devient un résidu de $10^{-16}$ et le test ne conclut plus.

**Indice.** Un seul $k$ annule la fonction objectif, et il correspond à la contrainte de trace $1 \cdot w_l + 3 \cdot w_\perp = 0$. Le balayage sert à vérifier que cette valeur est unique sur la grille, pas à la deviner.

La cellule s'exécute sans erreur tant que l'exercice n'est pas complété.


In [14]:
# Exercice 3 — localiser l'alignement
# TODO etudiant : balayer w_perp = Fraction(k, 12) pour k de -12 a 12,
# recalculer les six poids de la table, et imprimer le k pour lequel l'ecart
# maximal au facteur 2 est nul.
# Indice : la contrainte de trace sur C^4 (1*w_l + 3*w_perp = 0) fixe une seule
# valeur ; le balayage doit la retrouver sans qu'on la lui donne.
meilleur_k = None  # TODO etudiant : entier k dans [-12, 12]
if meilleur_k is not None:
    print("w_perp =", Fraction(meilleur_k, 12))
else:
    print("Exercice a completer")


Exercice a completer


## Périmètre honnête

Ce carnet mesure de la géométrie classique et de l'arithmétique de représentations. Il ne mesure rien qui dépende d'une expérience, et il ne prétend pas démontrer la proposition de Woit.

**Ce qui est mesuré ici.** Les structures complexes orthogonales de $\mathbb{R}^4$ et leurs deux composantes de chiralité ; la fibration de Hopf et sa fibre ; les espaces propres d'une structure fixée et la conjugaison qui les échange ; les poids de $\mathrm{U}(1)$ et la nécessité de la demi-puissance ; la dimension de $\mathrm{PT}$ ; la table d'une génération et le rapport constant à l'hypercharge.

**Ce qui est cité, non mesuré.** L'identification de $\mathrm{PT}$ au fibré des twisteurs au-dessus de $S^4$ et son rôle dans l'unification ; la quantification géométrique des orbites de $\mathrm{SU}(2,2)$ sur $\mathrm{PT}$, où la source signale des difficultés techniques (absence de polarisation invariante) ; la théorie $\mathcal{N}=4$ supersymétrique twistée que Woit discute en section 5 ; le traitement conjoint des signatures euclidienne et minkowskienne (section 4.3, par l'équateur de $S^4$ identifié à l'espace de Minkowski compactifié au temps $t = 0$). Aucun de ces points n'est vérifié par un calcul dans ce carnet.

**Les limites internes.** Le plancher numérique est celui de `float64` et `complex128`, voisin de $10^{-15}$ ; les résidus observés (`4.44e-16` à `1.22e-15`) sont à ce plancher, et un résidu de cet ordre ne distingue pas « exact » de « exact à l'arrondi près ». Le contre-témoin de la section 2(d) montre que deux familles de structures complexes coexistent dans $\mathbb{R}^4$ : le carnet en mesure une et nomme l'autre, sans prétendre qu'un choix de coordonnées est neutre. Enfin le facteur 2 de la section 5 est une convention d'hypercharge, pas un résultat.

**Pourquoi ces limites sont écrites.** Une mesure qui ne peut pas échouer ne prouve rien : chaque section en inclut donc une qui échoue. Mais un contre-témoin ne prouve pas non plus que l'énoncé positif est le bon — il prouve seulement que le contrôle sait détecter une erreur de la classe visée. Le reste est du ressort de la lecture de la source.


## À retenir

| Notion | Ce qu'elle dit | Où |
|---|---|---|
| Sphère des structures complexes | $aJ_1 + bJ_2 + cJ_3$ est une structure complexe ssi $a^2+b^2+c^2=1$ ; norme de Frobenius 2, déterminant 1 | §1 |
| Deux chiralités | l'espace des structures complexes orthogonales a deux composantes ; la carte de coordonnées produit l'autre, avec $\det = -1$ | §2(d) |
| Fibré de Hopf | $q \mapsto L(q) J_3 L(q)^{\mathsf T}$ atteint la sphère (forme close vérifiée sur 500 tirages), de fibre $q\,e^{k\theta}$ ; la phase scalaire $e^{i\theta}$ n'y est pas | §2 |
| Spineurs de Weyl | espaces propres $\pm i$, dimension complexe 2 chacun, échangés par la conjugaison | §3 |
| Demi-puissance forcée | poids $\{0,1,1,2\}$ asymétriques ; seul un décalage de $-1$ les symétrise | §3 |
| $\mathrm{PT} = \mathbb{CP}^3$ | $15 - 9 = 6 = 4 + 2$ ; le compte naïf $15 - 10 = 5$ est impair | §4 |
| Une génération | $2+1+1+6+3+3 = 16 = (1+3)(2+2)$ ; poids $= 2 \times$ hypercharge | §5 |
| Contrainte de trace | $w_l + 3w_\perp = 0$ fixe le secteur des quarks ; la violer casse trois rapports sur cinq | §5 |


## Vérifiez votre compréhension

**1.** Pourquoi $J = J_1 + 0{,}7\,J_2$ n'est-elle pas une structure complexe, alors que $J_1$ et $J_2$ le sont ?

<details><summary>Réponse</summary>

Parce que $J^2 = -(1 + 0{,}49)\,\mathbb{1} = -1{,}49\,\mathbb{1}$, donc $J^2 + \mathbb{1} = -0{,}49\,\mathbb{1}$ et le résidu vaut `0.490000` — exactement l'écart de $a^2+b^2+c^2$ à l'unité ($1{,}49 - 1$), et non une distance : la distance euclidienne de $(1;\,0{,}7;\,0)$ à la sphère vaut $0{,}22$. La condition $a^2 + b^2 + c^2 = 1$ n'est pas une normalisation cosmétique : c'est l'équation qui définit la sphère des structures complexes.

</details>

**2.** La phase $e^{k\theta}$ laisse le point de Hopf invariant ; pourquoi $e^{i\theta}$ ne le fait-elle pas, alors que les deux sont des cercles unitaires ?

<details><summary>Réponse</summary>

Parce que $k$ et $i$ ne commutent pas. Dans $q\,e^{k\theta}\,k\,e^{-k\theta}\bar q$ la phase se simplifie et l'on retrouve $q k \bar q$ ; dans $q\,e^{i\theta}\,k\,e^{-i\theta}\bar q$ elle ne se simplifie pas, et la mesure donne sa forme close : $\cos(2\theta)\,h_k(q) - \sin(2\theta)\,h_j(q)$, à `4.44e-16` près. L'écart vaut donc $|\sin 2\theta| \cdot |h_j - h_k|$, qui ne s'annule que pour les multiples de $\pi/2$. La mesure le confirme : écart `3.33e-16` pour la phase à droite, jusqu'à `1.97` pour la phase scalaire. La fibre du fibré est le sous-groupe qui **commute** avec la direction de base, pas un cercle quelconque.

</details>

**3.** Pourquoi le compte naïf de la dimension de $\mathrm{PT}$ donne-t-il 5, et pourquoi est-ce impossible ?

<details><summary>Réponse</summary>

Parce qu'il oublie la contrainte de trace du côté du stabilisateur. $\dim\mathfrak{su}(4) = 16 - 1 = 15$ et $\dim(\mathfrak{u}(1)\oplus\mathfrak{u}(3)) = 10$, mais le stabilisateur est $\mathrm{S}(\mathrm{U}(1)\times\mathrm{U}(3))$ : le déterminant y est aussi contraint, ce qui retire une dimension et donne 9. Le compte correct est $15 - 9 = 6$. Le nombre 5 est impair, donc ne peut pas être la dimension réelle d'une variété complexe — un signal d'erreur lisible sans rien savoir de la géométrie.

</details>

**4.** La table d'une génération donne un rapport constant de 2 à l'hypercharge. Que se passe-t-il si l'on prend $w_\perp = -1/2$ ?

<details><summary>Réponse</summary>

Les poids deviennent `[-1, -2, 0, 1/2, -1/2, 3/2]` et les rapports `[2, 2, 3, 3/2, 9/4]` : l'écart maximal au facteur 2 monte à `1.0`. La contrainte de trace $w_l + 3w_\perp = 0$ n'est donc pas décorative — c'est elle qui aligne le secteur des quarks. Trois rapports sur cinq sont cassés par une seule valeur fausse.

</details>


## Pour aller plus loin

- **Le carnet précédent de la série.** [Jauge-04](Jauge-04-Points-Spineurs-Twisteurs-Python.ipynb) construit la relation d'incidence des twisteurs, $\omega = iX\pi$, vérifiée sur 200 éléments de $\mathrm{SU}(2)$ (résidu maximal `1.43e-06`), et son exercice « tour complet » met en évidence le signe $-1$ du double revêtement $\mathrm{SU}(2) \to \mathrm{SO}(3)$. Ce carnet-ci part de la même géométrie, mais dans la signature euclidienne et du côté des structures complexes.
- **Les problèmes ouverts que la source nomme.** Woit range en section 5 : l'origine des générations (« it is unclear what the origin of generations might be »), la variété $S^7 \cong \mathrm{Spin}(8)/\mathrm{Spin}(7) = \mathrm{Spin}(6)/\mathrm{SU}(3)$ comme cadre possible — un fibré en cercles au-dessus de $\mathrm{PT}$ —, la quantification géométrique des orbites minimales de $\mathrm{SU}(2,2)$ sur $\mathrm{PT}$, où la source signale des difficultés techniques, et la théorie $\mathcal{N} = 4$ supersymétrique twistée. Aucun de ces points n'est traité ici.
- **Le chantier de distillation.** Ce carnet est la tranche Physique de l'issue [#28772](https://github.com/jsboige/CoursIA/issues/28772) : la transcription et la distillation de l'entretien de Curt Jaimungal avec Mikhail Gromov (23/11/2026), qui alimente les séries du Hub Sciences et calcul. Les autres tranches — Bio, Informatique, Catégories — se greffent sur leurs séries respectives.
- **Le reste de la sous-série.** [Jauge-01](Jauge-01-Theoreme-Noether-Python.ipynb) (Noether), [Jauge-02](Jauge-02-Invariance-Jauge-U1-SU2-Python.ipynb) (jauge locale), [Jauge-03](Jauge-03-Action-Wilson-Confinement-Python.ipynb) (Wilson sur réseau) : le fil va de la symétrie au réseau, puis à la géométrie des twisteurs.


## Références — copies archivées au gisement

- **P. Woit, _Euclidean Twistor Unification_**, arXiv:[2104.05099](https://arxiv.org/abs/2104.05099) (2021). Sections citées : 4.1 (structures complexes sur $\mathbb{R}^4$ et équation (4.1)), 4.2 (l'espace $\mathrm{PT}$ comme quotient, la fibration au-dessus de $S^4$, les poids $1$ et $-1/3$), 4.3 (relation entre les théories sur $\mathrm{PT}$ et l'espace-temps, signatures euclidienne et minkowskienne), 5 (problèmes ouverts : origine des générations, $S^7$, quantification géométrique, $\mathcal{N}=4$ twistée). Copie archivée : `G:\Mon Drive\MyIA\IA\Bibliographie IA\Physics\2021 - Woit - Euclidean Twistor Unification (arXiv 2104.05099).pdf`, empreinte md5 `8f0c5543993d4333f54c743e5ab4aed1`.
- **Les quaternions, le fibré de Hopf et les spineurs de Weyl** ne sont pas repris d'une source unique dans ce carnet : ils sont re-dérivés cellule par cellule, et chaque affirmation numérique a sa mesure. Le vocabulaire des spineurs de Weyl et le scindement $S_R \oplus S_L$ suivent la section 3, qui les construit par les espaces propres plutôt que par la théorie des représentations.
